# 18 (DS) — Time-Series Analytics

**Data Scientist perspective.** Temporal patterns end to end: daily rollups, month-over-month deltas, moving averages, gap detection and simple sessionization — all expressed as window functions over ISO dates, all pushed down to IRIS.

In [1]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

Connected to IRIS: <irispark.session.IrisParkSession object at 0xffff7469ff70>


In [2]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Event stream

120 days of orders across three sensors-like stores. ISO `YYYY-MM-DD` strings sort correctly, so string dates drive window ordering directly.

In [3]:
import random
import datetime as dt
import pandas as pd

random.seed(18)
base = dt.date(2025, 1, 1)
rows = []
for i in range(1, 901):
    d = (base + dt.timedelta(days=random.randint(0, 119))).isoformat()
    rows.append({
        "evento_id": i,
        "loja": random.choice(["norte", "sul", "centro"]),
        "data": d,
        "valor": round(random.uniform(5, 400), 2),
    })
eventos = session.createDataFrame(pd.DataFrame(rows))
print("events:", eventos.count())
eventos.show(3)

events: 900
   evento_id    loja        data   valor
0          1   norte  2025-01-24  266.27
1          2   norte  2025-02-12   83.19
2          3  centro  2025-03-04  200.14


## 2. Daily revenue rollup

In [4]:
from irispark.functions import sum as s

diario = eventos.groupBy("data").agg(s("valor").alias("receita")).orderBy("data")
diario.show(5)
print("active days:", diario.count())

         data  receita
0  2025-01-01  2005.86
1  2025-01-02  1127.17
2  2025-01-03  1070.19
3  2025-01-04  2302.64
4  2025-01-05   576.93
active days: 120


## 3. Moving average — 7-day frame

`rowsBetween(-6, 0)` over the date-ordered daily series gives a trailing weekly mean.

In [5]:
from irispark import Window
from irispark.functions import avg, col

w7 = Window.orderBy("data").rowsBetween(-6, 0)
media7 = diario.withColumn("media_movel_7d", avg(col("receita")).over(w7))
media7.orderBy("data DESC").show(7)

         data  receita  media_movel_7d
0  2025-04-30  1446.81     1272.084286
1  2025-04-29   765.92     1360.911429
2  2025-04-28  1568.68     1424.211429
3  2025-04-27  1321.34     1414.914286
4  2025-04-26   816.09     1465.658571
5  2025-04-25  1263.70     1541.772857
6  2025-04-24  1722.05     1491.581429


## 4. Month-over-month delta

Month key computed as a column first, then a monthly rollup; `lag` over the monthly window gives the previous value and the pct change.

In [6]:
from irispark.functions import lag, round as rnd, substring, col

ev2 = eventos.withColumn("mes", substring("data", 1, 7))
mensal = ev2.groupBy("mes").agg(s("valor").alias("receita"))
w_mes = Window.orderBy("mes")
mom = mensal.withColumn("mes_anterior", lag(col("receita"), 1).over(w_mes)) \
            .withColumn("variacao_pct",
                        rnd((col("receita") - col("mes_anterior")) / col("mes_anterior") * 100, 1)) \
            .orderBy("mes")
mom.show()

       mes   receita  mes_anterior variacao_pct
0  2025-01  47511.87           NaN         None
1  2025-02  46210.10      47511.87         -2.7
2  2025-03  51220.97      46210.10         10.8
3  2025-04  40273.84      51220.97        -21.4


## 5. Gap detection

Days between consecutive events per store: `datediff` of `lag`. Gaps above 2 days are surfaced.

In [7]:
from irispark.functions import datediff, when as w, lit

w_loja = Window.partitionBy("loja").orderBy("data")
gaps = eventos.distinct().withColumn("dia_antigo", lag(col("data"), 1).over(w_loja)) \
             .filter("dia_antigo IS NOT NULL") \
             .withColumn("dias_gap", datediff(col("data"), col("dia_antigo"))) \
             .withColumn("alerta", w(col("dias_gap") > 2, lit("GAP")).otherwise(lit("ok")))
gaps.groupBy("loja", "alerta").count().orderBy("loja", "alerta").show()
gaps.filter("alerta = 'GAP'").select("loja", "dia_antigo", "data", "dias_gap").show(5)

     loja alerta  count
0  CENTRO     ok    304
1   NORTE    GAP      1
2   NORTE     ok    289
3     SUL    GAP      1
4     SUL     ok    302
    loja  dia_antigo        data  dias_gap
0  NORTE  2025-03-12  2025-03-15         3
1    SUL  2025-01-04  2025-01-07         3


## 6. Recency deciles per store

Which stores went quiet recently? `ntile` over recency-ranked windows.

In [8]:
from irispark.functions import max as fmax, ntile, col

ultima = eventos.groupBy("loja").agg(fmax("data").alias("ultimo_dia"))
w_rec = Window.orderBy(col("ultimo_dia").desc())
recencia = ultima.withColumn("faixa_recencia", ntile(3).over(w_rec))
recencia.show()

     loja  ultimo_dia  faixa_recencia
0  CENTRO  2025-04-30               1
1   NORTE  2025-04-30               2
2     SUL  2025-04-30               3


## 7. SQL transparency

In [9]:
print(media7.to_sql())

SELECT *, AVG(receita) OVER (ORDER BY data ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS media_movel_7d FROM (SELECT * FROM (SELECT data, SUM(valor) AS receita FROM irispark_tmp_1_2c9fa5ff GROUP BY data) AS _g) AS _gc1 ORDER BY data


In [10]:
if session is not None:
    session.close()
    print("Session closed.")

Session closed.
